In [1]:
import csv
import pandas as pd
from os import path
import numpy as np
import matplotlib.pyplot as plt
import re
from scipy import interpolate
import numpy.typing as npt
from numpy.random import choice

In [2]:
pd.set_option('display.max_rows', 15)  # The number of rows above which it'll truncate the dataframe when displaying it. 
pd.set_option('display.min_rows', 10)  # The number of rows it'll be truncated to. 

### Reading in the total population for each area

In [22]:
df_county = pd.read_csv('data/geo_age_1918_round_male_county.csv')
df_county.columns = df_county.columns.str.lstrip('y')
df_county

,CountyID,County,Province,Population,0,1,2,3,4,5,...,96,97,98,99,100,101,102,103,104,105
0,CT00001,Akaroa,Canterbury,818,21,20,20,21,20,19,...,0,0,0,0,0,0,0,0,0,0
1,CT00002,Akitio,Wellington,644,17,17,16,16,15,15,...,0,0,0,0,0,0,0,0,0,0
2,CT00003,Amuri,Canterbury,1233,32,31,30,31,31,28,...,0,0,0,0,0,0,0,0,0,0
3,CT00004,Ashburton,Canterbury,5742,148,142,140,143,143,131,...,1,0,0,0,0,0,0,0,0,0
4,CT00005,Ashley,Canterbury,420,11,10,10,11,10,10,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,CT00122,Whakatane,Hawkes.Bay,1691,46,44,44,39,37,39,...,0,0,0,0,0,0,0,0,0,0
122,CT00123,Whangamomona,Taranaki,711,20,20,19,18,19,18,...,0,0,0,0,0,0,0,0,0,0
123,CT00124,Whangarei,Auckland,4369,116,116,110,116,108,109,...,1,0,0,0,0,0,0,0,0,0
124,CT00125,Whangaroa,Auckland,418,11,11,10,11,10,10,...,0,0,0,0,0,0,0,0,0,0


In [23]:
df_borough = pd.read_csv('data/geo_age_1918_round_male_borough.csv')
df_borough.columns = df_borough.columns.str.lstrip('y')
df_borough

,BoroughID,Borough,Population,Urban,Province,0,1,2,3,4,...,96,97,98,99,100,101,102,103,104,105
0,BR00001,Akaroa,273,Christchurch,Canterbury,6,6,5,6,5,...,0,0,0,0,0,0,0,0,0,0
1,BR00002,Alexandra,308,Dunedin,Otago,6,6,6,6,6,...,0,0,0,0,0,0,0,0,0,0
2,BR00003,Arrowtown,142,Dunedin,Otago,3,3,3,3,3,...,0,0,0,0,0,0,0,0,0,0
3,BR00004,Ashburton,2141,Christchurch,Canterbury,43,46,42,44,42,...,0,0,0,0,0,0,0,0,0,0
4,BR00005,Auckland,36139,Auckland,Auckland,695,708,673,710,720,...,7,1,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,BR00113,Whakatane,770,Gisborne,Hawkes.Bay,21,20,18,20,20,...,0,0,0,0,0,0,0,0,0,0
113,BR00114,Whangarei,1770,Auckland,Auckland,34,35,33,35,35,...,0,0,0,0,0,0,0,0,0,0
114,BR00115,Winton,276,Invercargill,Southland,6,6,6,6,6,...,0,0,0,0,0,0,0,0,0,0
115,BR00116,Woodville,520,PalmerstonNorth,Wellington,13,12,12,11,10,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df_locations_county = pd.read_csv('data/Geocodes_County_1918.csv')
df_locations_county

,CountyID,County,Latitude_South,Longitude_East
0,CT00001,Akaroa,-43.526945,172.632377
1,CT00002,Akitio,-40.600000,176.417000
2,CT00003,Amuri,-42.504808,173.022114
3,CT00004,Ashburton,-43.898340,171.730110
4,CT00005,Ashley,-43.303400,172.591400
...,...,...,...,...
121,CT00122,Whakatane,-37.958550,176.985450
122,CT00123,Whangamomona,-39.145245,174.738776
123,CT00124,Whangarei,-35.731670,174.323910
124,CT00125,Whangaroa,-35.052055,173.743462


In [25]:
df_locations_borough = pd.read_csv('data/Geocodes_Borough_1918.csv')
df_locations_borough

,BoroughID,Borough,Latitude_South,Longitude_East
0,BR00001,Akaroa,-43.80467,172.966667
1,BR00002,Alexandra,-45.24917,169.379720
2,BR00003,Arrowtown,-44.94250,168.835830
3,BR00004,Ashburton,-43.90556,171.745560
4,BR00005,Auckland,-36.86667,174.766670
...,...,...,...,...
112,BR00113,Whakatane,-37.95850,176.985000
113,BR00114,Whangarei,-35.73167,174.323910
114,BR00115,Winton,-46.15000,168.333330
115,BR00116,Woodville,-40.33710,175.866700


In [32]:

def combine_demography(df_county, df_borough, df_locations_county, df_locations_borough):
    """Combine county and borough demography dataframes with their location data.

    Returns a single DataFrame with columns:
        ID, Name, Latitude, Longitude, 0, 1, ..., 105
    """
    year_cols = [str(i) for i in range(106)]

    # Merge county demography with locations
    county = df_county.merge(df_locations_county[['CountyID', 'Latitude_South', 'Longitude_East']],
                             on='CountyID', how='left')
    county = county.rename(columns={
        'CountyID': 'ID',
        'County': 'Name',
        'Latitude_South': 'Latitude',
        'Longitude_East': 'Longitude',
    })
    county = county[['ID', 'Name', 'Latitude', 'Longitude','Province'] + year_cols]

    # Merge borough demography with locations (strip any whitespace from column names first)
    df_locations_borough.columns = df_locations_borough.columns.str.strip()
    borough = df_borough.merge(df_locations_borough[['BoroughID', 'Latitude_South', 'Longitude_East']],
                               on='BoroughID', how='left')
    borough = borough.rename(columns={
        'BoroughID': 'ID',
        'Borough': 'Name',
        'Latitude_South': 'Latitude',
        'Longitude_East': 'Longitude',
    })
    borough = borough[['ID', 'Name', 'Latitude', 'Longitude','Province'] + year_cols]

    combined = pd.concat([county, borough], ignore_index=True)
    return combined

In [33]:
df_combined = combine_demography(df_county, df_borough, df_locations_county, df_locations_borough)
df_combined

,ID,Name,Latitude,Longitude,Province,0,1,2,3,4,...,96,97,98,99,100,101,102,103,104,105
0,CT00001,Akaroa,-43.526945,172.632377,Canterbury,21,20,20,21,20,...,0,0,0,0,0,0,0,0,0,0
1,CT00002,Akitio,-40.600000,176.417000,Wellington,17,17,16,16,15,...,0,0,0,0,0,0,0,0,0,0
2,CT00003,Amuri,-42.504808,173.022114,Canterbury,32,31,30,31,31,...,0,0,0,0,0,0,0,0,0,0
3,CT00004,Ashburton,-43.898340,171.730110,Canterbury,148,142,140,143,143,...,1,0,0,0,0,0,0,0,0,0
4,CT00005,Ashley,-43.303400,172.591400,Canterbury,11,10,10,11,10,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,BR00113,Whakatane,-37.958500,176.985000,Hawkes.Bay,21,20,18,20,20,...,0,0,0,0,0,0,0,0,0,0
239,BR00114,Whangarei,-35.731670,174.323910,Auckland,34,35,33,35,35,...,0,0,0,0,0,0,0,0,0,0
240,BR00115,Winton,-46.150000,168.333330,Southland,6,6,6,6,6,...,0,0,0,0,0,0,0,0,0,0
241,BR00116,Woodville,-40.337100,175.866700,Wellington,13,12,12,11,10,...,0,0,0,0,0,0,0,0,0,0


In [28]:
#df_combined.to_csv('clean_data/demography_male.csv')

In [34]:
df_combined[['ID','Name', 'Province','Latitude', 'Longitude']].to_csv('clean_data/coord_SGU.csv')

In [41]:
df_provinces = df_combined.groupby('Province')[['Latitude','Longitude']].mean()
#df_provinces.to_csv('clean_data/coord_Province.csv')